<a href="https://colab.research.google.com/github/JosuePerezValenzuela/WebScraping2/blob/master/Copia_de_WebScraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

WebScripng


In [2]:
import requests
import pandas as pd
headers={"USER_AGENT":"Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko)Chrome/44.0.2403.157 Safari/537.36", "origin":"https://www.premierleague.com"}
equipos_totales=[]
for i in range(0,2):
 url_api='http://footballapi.pulselive.com/football/fixtures?comps=1&compSeasons=42&teams=1,2,127,4,6,7,26,10,11,12,23,14,20,42,29,45,21,33,36,25&pageSize=40&sort=desc&statuses=C&altIds=true&page='+ str(i)
 response=requests.get(url_api,headers=headers)
 data=response.json()
 partidos=data["content"]
 for partido in partidos:
    equipos_totales.append(
    {
        "score": partido["teams"][0]["score"],
        "name": partido["teams"][0]["team"]["name"]
    })
    equipos_totales.append(
    {
        "score": partido["teams"][1]["score"],
        "name": partido["teams"][1]["team"]["name"]
    })
df=pd.DataFrame(equipos_totales)
print(df)


     score               name
0      3.0  Manchester United
1      1.0        Bournemouth
2      4.0            Arsenal
3      0.0        Aston Villa
4      1.0            Chelsea
..     ...                ...
155    1.0  Manchester United
156    3.0  Tottenham Hotspur
157    0.0        Bournemouth
158    1.0   Newcastle United
159    1.0         Sunderland

[160 rows x 2 columns]


WebScriping los tiempos

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# URL de la página de noticias
url = 'https://www.lostiempos.com/ultimas-noticias'

# Encabezados HTTP para simular un navegador
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8',
    'Accept-Encoding': 'gzip, deflate, br',
    'Accept-Language': 'es-ES,es;q=0.9,en;q=0.8',
    'Connection': 'keep-alive',
    'Upgrade-Insecure-Requests': '1',
    'Referer': 'https://www.google.com/'
}

# Realiza la solicitud HTTP a la página web
response = requests.get(url, headers=headers)

news_data = []  # Lista para almacenar los datos

if response.status_code == 200:
    # Analiza el contenido de la página
    soup = BeautifulSoup(response.content, 'html.parser')

    # Selecciona todos los elementos que contienen cada noticia
    news_items = soup.select('div.views-row.noticia-lt')

    # Recorre cada noticia y extrae la información
    for idx, item in enumerate(news_items, start=1):
        # Extraer título y enlace
        title_elem = item.select_one('.views-field-title a')
        title = title_elem.get_text(strip=True) if title_elem else 'No title'
        link = title_elem['href'] if title_elem and 'href' in title_elem.attrs else 'No link'
        if link.startswith('/'):
            link = 'https://www.lostiempos.com' + link

        # Extraer fecha
        date_elem = item.select_one('.views-field-field-noticia-fecha .date-display-single')
        date_text = date_elem.get_text(strip=True) if date_elem else 'No date'

        # Extraer categoría
        category_elem = item.select_one('.views-field-seccion')
        category = category_elem.get_text(strip=True) if category_elem else 'No category'

        # Extraer resumen
        summary_elem = item.select_one('.views-field-field-noticia-sumario')
        summary = summary_elem.get_text(strip=True) if summary_elem else 'No summary'

        # Extraer URL de la imagen
        image_elem = item.select_one('.views-field-field-noticia-fotos img')
        image_url = image_elem['src'] if image_elem and 'src' in image_elem.attrs else 'No image'

        # Almacenar en la lista de diccionarios
        news_data.append({
            'Título': title,
            'Enlace': link,
            'Fecha': date_text,
            'Categoría': category,
            'Resumen': summary,
            'Imagen': image_url
        })

        # Imprimir la noticia en formato texto
        print(f"Noticia {idx}")
        print(f"Título: {title}")
        print(f"Enlace: {link}")
        print(f"Fecha: {date_text}")
        print(f"Categoría: {category}")
        print(f"Resumen: {summary}")
        print(f"Imagen: {image_url}")
        print("-" * 80)

    # Crear un DataFrame con los datos extraídos y mostrarlo
    news_df = pd.DataFrame(news_data)
    print("\n--- DataFrame con las noticias ---")
    print(news_df)
else:
    print("Error al acceder a la página:", response.status_code)




Noticia 1
Título: Sedes reporta la muerte de un adolescente por hepatitis
Enlace: https://www.lostiempos.com/actualidad/cochabamba/20250408/sedes-reporta-muerte-adolescente-hepatitis
Fecha: 08/04/2025 - 14:02
Categoría: Cochabamba
Resumen: El Sedes de Cochabamba informó hoy que un adolescente de 14 años de edad falleció por hepatitis y recomendó a los padres que acudan a los centros de salud más cercanos en casos de presentar síntomas, como la fiebre.
Imagen: https://www.lostiempos.com/sites/default/files/styles/noticia_home_tipo_1/public/media_imagen/2025/4/8/hepatitis_imagen.jpg?itok=95N2_iWy
--------------------------------------------------------------------------------
Noticia 2
Título: ADN afirma que Jaime Dunn será su candidato a la presidencia, pero este aún no lo confirma
Enlace: https://www.lostiempos.com/actualidad/pais/20250408/adn-afirma-que-jaime-dunn-sera-su-candidato-presidencia-pero-este-aun-no
Fecha: 08/04/2025 - 13:30
Categoría: País
Resumen: Desde Acción Democrática

In [4]:
import sqlite3
import pandas as pd
import requests
headers={"USER_AGENT":"Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko)Chrome/44.0.2403.157 Safari/537.36", "origin":"https://www.premierleague.com"}
equipos_totales=[]
for i in range(0,2):
 url_api='http://footballapi.pulselive.com/football/fixtures?comps=1&compSeasons=42&teams=1,2,127,4,6,7,26,10,11,12,23,14,20,42,29,45,21,33,36,25&pageSize=40&sort=desc&statuses=C&altIds=true&page='+ str(i)
 response=requests.get(url_api,headers=headers)
 data=response.json()
 partidos=data["content"]
 for partido in partidos:
    equipos_totales.append(
    {
        "score": partido["teams"][0]["score"],
        "name": partido["teams"][0]["team"]["name"]
    })
    equipos_totales.append(
    {
        "score": partido["teams"][1]["score"],
        "name": partido["teams"][1]["team"]["name"]
    })
df=pd.DataFrame(equipos_totales)
print(df)




     score               name
0      3.0  Manchester United
1      1.0        Bournemouth
2      4.0            Arsenal
3      0.0        Aston Villa
4      1.0            Chelsea
..     ...                ...
155    1.0  Manchester United
156    3.0  Tottenham Hotspur
157    0.0        Bournemouth
158    1.0   Newcastle United
159    1.0         Sunderland

[160 rows x 2 columns]


In [5]:
# Crear una conexión a la base de datos SQLite
conn = sqlite3.connect('df.db')

# Guardar el DataFrame en la base de datos como una tabla
df.to_sql('score', conn, if_exists='replace', index=False)

# Cerrar la conexión a la base de datos
# Crear un cursor para ejecutar comandos SQL
cursor = conn.cursor()

# Ejecutar una consulta SELECT en la tabla ''
cursor.execute("SELECT name, count(1) FROM score WHERE name like'%United%' GROUP By name")

# Obtener los resultados de la consulta
resultados = cursor.fetchall()

# Cerrar la conexión a la base de datos
conn.close()

# Mostrar los resultados
for row in resultados:
    print(row)

conn.close()

('Manchester United', 9)
('Newcastle United', 9)
('West Ham United', 8)


# Imagen to Texto

In [6]:
# instalar librerias
!pip install pytesseract
!sudo apt-get install tesseract-ocr
!pip install pillow

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  tesseract-ocr-eng tesseract-ocr-osd
The following NEW packages will be installed:
  tesseract-ocr tesseract-ocr-eng tesseract-ocr-osd
0 upgraded, 3 newly installed, 0 to remove and 30 not upgraded.
Need to get 4,816 kB of archives.
After this operation, 15.6 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-eng all 1:4.00~git30-7274cfa-1.1 [1,591 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-osd all 1:4.00~git30-7274cfa-1.1 [2,990 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr amd64 4.1.1-2.1build1 [236 kB]
Fetched 4,816 kB in 2s (1,932 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debc

In [7]:
# Obtener la URL de la imagen
url = "https://www.minedu.gob.bo/images/WhatsApp_Image_2020-07-03_at_095025.jpeg"
import pytesseract
from PIL import Image
import requests
from io import BytesIO

# Function to extract text from an image URL
def extract_text_from_image_url(image_url):
    # Download the image from the URL
    response = requests.get(image_url)
    img = Image.open(BytesIO(response.content))

    # Use pytesseract to extract text from the image
    text = pytesseract.image_to_string(img)

    return text

# Replace this with the URL of the image you want to process
image_url = url

# Call the function to extract text from the image URL
extracted_text = extract_text_from_image_url(image_url)

# Print the extracted text
print(extracted_text)

  

4

Gobierno del Estado Plu Gobierno del Estado

BOLIV! BOLI

Ministerio de Educacién, Ministerio de Salud
Deportes y Culturas

 

 

   

   

COMUNICADO BI-MINISTERIAL

El Ministerio de Educacién, Deportes y Culturas y el Ministerio de Salud, comunican a la
opinion publica que, segun proyecciones elaboradas recientemente en base a metodologia
cientifica, se estima que el pico mas alto de la pandemia se presentara entre la Ultima
semana del mes de agosto y la primera semana de septiembre, lo que pone en alto riesgo la
posibilidad de que los estudiantes retornen a sus labores escolares presenciales.

El retorno a clases en la modalidad semipresencial 0 presencial implicaria un aumento al
doble o triple de infectados y podria provocar el colapso de los sistemas sanitarios a nivel
nacional.

Por lo tanto, a fin de precautelar la salud de los estudiantes, maestros, personal
administrativo de las Unidades Educativas, padres de familia y poblacién de todo el pais, el
Ministerio de Educac

# Audio to Texto

In [8]:
# Instalar Libreria
!pip install SpeechRecognition
!pip install pydub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 49.1 MB/s eta 0:00:00


In [9]:
audio_url = 'https://audio.conciencia.net/2023/2023nov30.mp3'
audio_url = 'https://audio.conciencia.net/2023/2023dic01.mp3'
import speech_recognition as sr
import requests
from io import BytesIO
from pydub import AudioSegment

# Crea un objeto de reconocimiento de voz
recognizer = sr.Recognizer()

# Descarga el archivo de audio desde la URL
response = requests.get(audio_url)
audio_data = BytesIO(response.content)

# Convierte el archivo MP3 a formato WAV
audio = AudioSegment.from_mp3(audio_data)
audio.export("audio.wav", format="wav")
text = ''
# Transcribe el contenido del archivo de audio WAV
with sr.AudioFile("audio.wav") as source:
    try:
        audio = recognizer.record(source)
        text = text+ recognizer.recognize_google(audio, language="es-ES")

    except sr.UnknownValueError:
        print("No se pudo reconocer el audio.")
    except sr.RequestError as e:
        print(f"Error en la solicitud: {e}")
print("Texto transcribido:")
print(text)

Texto transcribido:
un mensaje a la conciencia un momento de reflexión en la vida diaria escúchelo hoy en la voz de Carlos Rey Lencho patas planas mítico personaje creado por David Pinto es un vaquero de esos de los viejos tiempos de Oriente cuando ese desierto de Guatemala fue poblado por españoles y sus descendientes vieron en sus áridas tierras un reflejo de las tierras lejanas que habían dejado para establecerse en el Nuevo Mundo aunque ahora vive lejos de su tierra en el frío y recatado Occidente Lencho mantiene la chispa y la generosidad de los habitantes del rudo y a primera vista inhóspito desierto oriental guatemalteco Lencho es un cuentacuentos en él sobreviven las historias más fantásticas y representativas de la cultura oriental del país que suelta con tal naturalidad que los oyentes nunca saben si son falsas o verdaderas Lencho se saca cuentos de la cabeza como un marchante saca su mercancía de un costal una vez cuenta Lencho su pueblo natal la polvorienta comunidad de Ipa

# Video To Texto

In [10]:
# instalar Librerias
!pip install moviepy

In [11]:
video_url = 'https://video.conciencia.net/2023/2023dic01.mp4'

import requests
from moviepy.editor import VideoFileClip
import speech_recognition as sr

# Descargar el video desde la URL
response = requests.get(video_url)
video_data = response.content

# Guardar el video descargado en un archivo local
with open('video.mp4', 'wb') as f:
    f.write(video_data)

# Extraer el audio del video
video = VideoFileClip('video.mp4')
audio = video.audio

# Crear un objeto de reconocimiento de voz
recognizer = sr.Recognizer()

# Transcribir el audio del video
with sr.AudioFile('audio.wav') as source:
    try:
        audio_data = recognizer.record(source)
        text = recognizer.recognize_google(audio_data, language="es-ES")
        print("Texto transcribido:")
        print(text)
    except sr.UnknownValueError:
        print("No se pudo reconocer el audio.")
    except sr.RequestError as e:
        print(f"Error en la solicitud: {e}")

Texto transcribido:
un mensaje a la conciencia un momento de reflexión en la vida diaria escúchelo hoy en la voz de Carlos Rey Lencho patas planas mítico personaje creado por David Pinto es un vaquero de esos de los viejos tiempos de Oriente cuando ese desierto de Guatemala fue poblado por españoles y sus descendientes vieron en sus áridas tierras un reflejo de las tierras lejanas que habían dejado para establecerse en el Nuevo Mundo aunque ahora vive lejos de su tierra en el frío y recatado Occidente Lencho mantiene la chispa y la generosidad de los habitantes del rudo y a primera vista inhóspito desierto oriental guatemalteco Lencho es un cuentacuentos en él sobreviven las historias más fantásticas y representativas de la cultura oriental del país que suelta con tal naturalidad que los oyentes nunca saben si son falsas o verdaderas Lencho se saca cuentos de la cabeza como un marchante saca su mercancía de un costal una vez cuenta Lencho su pueblo natal la polvorienta comunidad de Ipa

In [12]:
import requests
from moviepy.editor import VideoFileClip
import speech_recognition as sr

# URL del video
video_url = 'https://video.conciencia.net/2023/2023nov30.mp4'

# Descargar el video desde la URL
response = requests.get(video_url)
video_data = response.content

# Guardar el video descargado en un archivo local
with open('video1.mp4', 'wb') as f:
    f.write(video_data)

# Extraer el audio del video
video = VideoFileClip('video1.mp4')
audio = video.audio

# Asegurarse de que se extraiga todo el audio del video
audio = audio.set_duration(video.duration)

# Crear un objeto de reconocimiento de voz
recognizer = sr.Recognizer()

# Transcribir el audio del video
with sr.AudioFile('audio.wav') as source:
    try:
        audio_data = recognizer.record(source)
        text = recognizer.recognize_google(audio_data, language="es-ES")
        print("Texto transcribido:")
        print(text)
    except sr.UnknownValueError:
        print("No se pudo reconocer el audio.")
    except sr.RequestError as e:
        print(f"Error en la solicitud: {e}")

Texto transcribido:
un mensaje a la conciencia un momento de reflexión en la vida diaria escúchelo hoy en la voz de Carlos Rey Lencho patas planas mítico personaje creado por David Pinto es un vaquero de esos de los viejos tiempos de Oriente cuando ese desierto de Guatemala fue poblado por españoles y sus descendientes vieron en sus áridas tierras un reflejo de las tierras lejanas que habían dejado para establecerse en el Nuevo Mundo aunque ahora vive lejos de su tierra en el frío y recatado Occidente Lencho mantiene la chispa y la generosidad de los habitantes del rudo y a primera vista inhóspito desierto oriental guatemalteco Lencho es un cuentacuentos en él sobreviven las historias más fantásticas y representativas de la cultura oriental del país que suelta con tal naturalidad que los oyentes nunca saben si son falsas o verdaderas Lencho se saca cuentos de la cabeza como un marchante saca su mercancía de un costal una vez cuenta Lencho su pueblo natal la polvorienta comunidad de Ipa